# BGK PLN — MVP dashboard

Pierwszy MVP. Trzy sekcje, wszystkie PLN programy:

1. **Outstanding timeline** — per-program (FPC / KFD / FP / FWSZ / własne), mld PLN, z `v_bgk_outstanding_timeline`.
2. **Spread vs POLGB** — per-issuance spread w bp, z `v_bgk_issuance_spread` (Milestone B, all-PLN coverage). Osobne panele dla fixed-coupon (yield-space) i floater (DM-space, z coupon margin z XLSX).
3. **Recent issuances** — ostatnie 20 emisji ze spreadem.

Źródło danych: Supabase project shared z CETO_DOWNLOADER.

**Setup:** `pip install -r requirements-notebook.txt --trusted-host pypi.org --trusted-host files.pythonhosted.org`, ustaw `SUPABASE_URL` i `SUPABASE_SERVICE_ROLE_KEY` w `.env` lub w shell przed `jupyter lab`.

In [1]:
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import requests
from IPython.display import display

# Load .env if present; else rely on shell-set env vars.
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env')
except ImportError:
    pass

pio.renderers.default = 'notebook_connected'

SUPABASE_URL = os.environ['SUPABASE_URL'].rstrip('/')
# Normalise: tolerate operator pasting the PostgREST endpoint by mistake.
for suffix in ('/rest/v1', '/rest'):
    if SUPABASE_URL.endswith(suffix):
        SUPABASE_URL = SUPABASE_URL[: -len(suffix)]
        break
SUPABASE_KEY = os.environ['SUPABASE_SERVICE_ROLE_KEY']

HEADERS = {
    'apikey': SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
    'Content-Type': 'application/json',
}
PAGE_SIZE = 1000


def fetch_view(name: str, query: str = '?select=*') -> pd.DataFrame:
    """GET a PostgREST view/table, paginating until exhausted."""
    rows: list = []
    for page in range(200):
        offset = page * PAGE_SIZE
        h = {**HEADERS, 'Range-Unit': 'items',
             'Range': f'{offset}-{offset + PAGE_SIZE - 1}'}
        r = requests.get(f'{SUPABASE_URL}/rest/v1/{name}{query}',
                         headers=h, timeout=60)
        if r.status_code not in (200, 206):
            r.raise_for_status()
        chunk = r.json()
        rows.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            break
    return pd.DataFrame(rows)


print(f'Connected to {SUPABASE_URL}')

Connected to https://tclcprvelrzkaichgjgd.supabase.co


## 1. Outstanding PLN per program — running total over time

Step chart per program (FPC / KFD / FP / FWSZ / własne). Każda emisja podbija outstanding programu, każdy wykup go obniża. Z `v_bgk_outstanding_timeline` przefiltrowane do `currency='PLN'`.

In [ ]:
df_timeline = fetch_view(
    'v_bgk_outstanding_timeline',
    '?currency=eq.PLN&select=*&order=event_date.asc',
)
df_timeline['event_date'] = pd.to_datetime(df_timeline['event_date'])
df_timeline['outstanding_bn'] = df_timeline['running_outstanding'].astype(float) / 1e9

# Order programs by current outstanding (descending) so legend reflects size.
program_order = (df_timeline.groupby('program')['outstanding_bn'].last()
                 .sort_values(ascending=False).index.tolist())

fig = go.Figure()
for prog in program_order:
    sub = df_timeline[df_timeline['program'] == prog].sort_values('event_date')
    fig.add_trace(go.Scatter(
        x=sub['event_date'],
        y=sub['outstanding_bn'],
        mode='lines',
        line_shape='hv',
        name=prog,
        hovertemplate=f'{prog}<br>%{{x|%Y-%m-%d}}<br>%{{y:.2f}} mld PLN<extra></extra>',
    ))

fig.update_layout(
    title='BGK PLN outstanding per program (mld PLN)',
    xaxis_title=None,
    yaxis_title='Outstanding (mld PLN)',
    hovermode='x unified',
    height=460,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

latest = (df_timeline.sort_values('event_date').groupby('program')
          .agg(last_event=('event_date', 'last'),
               outstanding_bn=('outstanding_bn', 'last'))
          .sort_values('outstanding_bn', ascending=False))
print('\nLatest outstanding per program:')
display(latest.style.format({'outstanding_bn': '{:.2f} mld PLN'}))

## 2. Spread vs POLGB curve — per issuance, all PLN programs

Per-issuance spread w bp z `v_bgk_issuance_spread`. Dwa panele:

- **Top — fixed-coupon bonds** (yield-space): `bgk_yield_pct - polgb_yield_interp(date, tenor)` × 100 bp. Pozytywny spread = BGK pays premium nad POLGB.
- **Bottom — floater bonds** (DM-space): `(bgk_true_DM - polgb_WZ_DM_interp) × 100` bp, gdzie `bgk_true_DM = price-implied + coupon_margin` (margin sparsowany z XLSX "Kupon", np. WIBOR6M + 0,50%).

Colored per series. FPC1140 (~14Y) zostaje NULL bo poza zakresem WZ curve.

In [ ]:
from plotly.subplots import make_subplots

df_spread = fetch_view(
    'v_bgk_issuance_spread',
    '?spread_bp=not.is.null&select=*&order=issue_date.asc',
)
df_spread['issue_date'] = pd.to_datetime(df_spread['issue_date'])
for c in ['spread_bp', 'tenor_years', 'bgk_yield_pct', 'bgk_true_dm_pct',
          'bgk_margin_bp', 'bgk_price_pct']:
    if c in df_spread.columns:
        df_spread[c] = pd.to_numeric(df_spread[c], errors='coerce')

fixed = df_spread[df_spread['bgk_coupon_kind'] == 'stałe'].copy()
floater = df_spread[df_spread['bgk_coupon_kind'] == 'zmienne'].copy()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        f'Fixed-coupon spread vs POLGB nominal curve (bp) — {len(fixed)} emisji',
        f'Floater spread vs POLGB WZ curve (bp, DM-space) — {len(floater)} emisji',
    ),
)

# Top panel - fixed coupon
for series in sorted(fixed['series'].unique()):
    sub = fixed[fixed['series'] == series].sort_values('issue_date')
    fig.add_trace(go.Scatter(
        x=sub['issue_date'], y=sub['spread_bp'],
        mode='lines+markers', name=series, legendgroup='fixed',
        customdata=sub[['program', 'tenor_years']].values,
        hovertemplate=(f'{series} (%{{customdata[0]}})<br>'
                       '%{x|%Y-%m-%d}<br>spread: %{y:.1f} bp<br>'
                       'tenor: %{customdata[1]:.1f} y<extra></extra>'),
    ), row=1, col=1)

# Bottom panel - floaters
for series in sorted(floater['series'].unique()):
    sub = floater[floater['series'] == series].sort_values('issue_date')
    fig.add_trace(go.Scatter(
        x=sub['issue_date'], y=sub['spread_bp'],
        mode='lines+markers', name=series, legendgroup='floater',
        customdata=sub[['program', 'tenor_years', 'bgk_margin_bp']].values,
        hovertemplate=(f'{series} (%{{customdata[0]}})<br>'
                       '%{x|%Y-%m-%d}<br>spread: %{y:.1f} bp<br>'
                       'tenor: %{customdata[1]:.1f} y<br>'
                       'coupon margin: %{customdata[2]:.0f} bp<extra></extra>'),
    ), row=2, col=1)

fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4, row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4, row=2, col=1)
fig.update_yaxes(title_text='Spread (bp)', row=1, col=1)
fig.update_yaxes(title_text='Spread (bp)', row=2, col=1)
fig.update_layout(
    height=720, template='plotly_white', hovermode='x unified',
    legend=dict(orientation='v', yanchor='top', y=1, xanchor='left', x=1.02),
)
fig.show()

summary = (df_spread.groupby(['bgk_coupon_kind', 'series'])['spread_bp']
           .agg(['count', 'mean', 'min', 'max']).round(1)
           .rename(columns={'count': 'n', 'mean': 'avg_bp', 'min': 'min_bp', 'max': 'max_bp'}))
print('\nSpread summary per series:')
display(summary)

## 3. Recent issuances — last 20 across all PLN programs

Per-emisja snapshot z `v_bgk_issuance_spread`. Floatery pokazują `bgk_true_dm` (price + margin), fixed pokazują `bgk_yield`. Spread vs POLGB krzywej w prawej kolumnie.

In [ ]:
df_recent = fetch_view(
    'v_bgk_issuance_spread',
    '?select=issue_date,program,series,isin,tenor_years,bgk_coupon_kind,'
    'bgk_margin_bp,bgk_yield_pct,bgk_true_dm_pct,bgk_price_pct,issue_amount,spread_bp'
    '&order=issue_date.desc&limit=20',
)
for c in ['tenor_years', 'bgk_margin_bp', 'bgk_yield_pct', 'bgk_true_dm_pct',
          'bgk_price_pct', 'issue_amount', 'spread_bp']:
    df_recent[c] = pd.to_numeric(df_recent[c], errors='coerce')

# Show yield for fixed bonds, true_DM for floaters in one combined column
df_recent['rate_pct'] = df_recent['bgk_yield_pct'].fillna(df_recent['bgk_true_dm_pct'])
df_recent['issue_amount_mln'] = df_recent['issue_amount'] / 1e6

df_show = df_recent.rename(columns={
    'issue_date': 'date',
    'bgk_coupon_kind': 'kind',
    'bgk_margin_bp': 'margin_bp',
    'bgk_price_pct': 'price',
    'tenor_years': 'tenor_y',
})[['date', 'program', 'series', 'isin', 'kind', 'tenor_y',
    'margin_bp', 'price', 'rate_pct', 'issue_amount_mln', 'spread_bp']]

display(df_show.style.format({
    'tenor_y':          '{:.2f}',
    'margin_bp':        '{:.0f}',
    'price':            '{:.3f}',
    'rate_pct':         '{:.3f}',
    'issue_amount_mln': '{:,.0f}',
    'spread_bp':        '{:.1f}',
}, na_rep='—').hide(axis='index'))